In [ ]:
import os

INPUT_BASE = '/kaggle/input/datasets/msebastiaanh/7gcn-yeet/HAABSA7GCN'

assert os.path.isdir(f'{INPUT_BASE}/src'), f"src/ not found under {INPUT_BASE}"
assert os.path.isdir(f'{INPUT_BASE}/data'), f"data/ not found under {INPUT_BASE}"
print('src/:', sorted(os.listdir(f'{INPUT_BASE}/src')))
print('data/:', sorted(os.listdir(f'{INPUT_BASE}/data')))

In [ ]:
import shutil, os, sys

WORK = '/kaggle/working'
os.makedirs(f'{WORK}/data', exist_ok=True)

# Copy all data files (.const, .dep, ontology, rem*.csv, cross_features_*.pt)
for f in os.listdir(f'{INPUT_BASE}/data'):
    shutil.copy(f'{INPUT_BASE}/data/{f}', f'{WORK}/data/{f}')

shutil.copy(f'{INPUT_BASE}/cooc_matrix_final2.csv', f'{WORK}/cooc_matrix_final2.csv')
shutil.copy(f'{INPUT_BASE}/src/test_ontology_keys.csv', f'{WORK}/test_ontology_keys.csv')

# 6GCN: cross_example.py and fusion.py are IMPORTED (not exec'd) by the notebook
# cells, so they must be on sys.path in the working dir.
for m in ['cross_example.py', 'fusion.py']:
    shutil.copy(f'{INPUT_BASE}/src/{m}', f'{WORK}/{m}')
sys.path.insert(0, WORK)

os.chdir(WORK)
print('cwd:', os.getcwd())
print('working/ contents:', sorted(os.listdir(WORK)))

# verify const files
for year in ['2015', '2016']:
    for split in ['train', 'test']:
        p = f'data/{split}{year}restaurant.txt.const'
        assert os.path.exists(p), f'MISSING: {p}'

# verify 6GCN dependencies landed in working dir
for year in ['2015', '2016']:
    assert os.path.exists(f'data/cross_features_{year}.pt'), f'cross_features_{year}.pt missing in working'
assert os.path.exists('cross_example.py') and os.path.exists('fusion.py'), 'module files missing in working'
print('const + cooc + ontology + cross_features + modules OK')

In [ ]:
!pip install pytorch_transformers==1.2.0 --quiet

In [ ]:
import torch, sys
print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
import pytorch_transformers
print('pytorch_transformers:', pytorch_transformers.__version__)

In [ ]:
import nbformat
NB_PATH = f'{INPUT_BASE}/src/7GCN.ipynb'
nb = nbformat.read(NB_PATH, as_version=4)

DEFINITION_CELLS = list(range(0, 25))
for i, cell in enumerate(nb.cells):
    if i not in DEFINITION_CELLS or cell.cell_type != 'code':
        continue
    src = ''.join(cell.source)
    if not src.strip():
        continue
    try:
        exec(src, globals())
        print(f"✓ cell {i:2d} OK ({len(src)} chars)")
    except Exception as e:
        print(f"✗ cell {i:2d} FAILED: {type(e).__name__}: {e}")
        raise

In [ ]:
import pandas as pd
print('Loading co-occurrence matrix...')
cooc = pd.read_csv('cooc_matrix_final2.csv', index_col=0)
print(f'  cooc shape: {cooc.shape}')

print('Loading ontology words...')
onto_words_df = pd.read_csv('test_ontology_keys.csv', sep=';')
onto_words = [item for sublist in onto_words_df.values.tolist() for item in sublist]
onto_words = list(dict.fromkeys(onto_words))
print(f'  ontology words count: {len(onto_words)}')

In [ ]:
import os, json, time, gc

EPOCHS = 15        
SEED   = 7
YEAR   = '2016'     
XSIM_TOP_K = 20 
CONCAT_DROPOUT_FIXED = 0.2285714285714286
OUTDIR = '/kaggle/working/top3_7gcn_results'
os.makedirs(OUTDIR, exist_ok=True)

# top-3 from the final 13-trial standings (val-sorted)
CONFIGS = [
    {'rank': 1, 'lr': 1.0490678654396277e-05, 'dropout': 0.20,
     'l2reg': 0.0014773338109036142, 'fusion_dropout': 0.30},   # search val 0.9216
    #{'rank': 2, 'lr': 8.836537235645673e-06,  'dropout': 0.25,
     #'l2reg': 0.0005059218096160434, 'fusion_dropout': 0.00},   # seeded; search val 0.9176
    #{'rank': 3, 'lr': 1.150150314823389e-05,  'dropout': 0.35,
     #'l2reg': 0.0025650320453225314, 'fusion_dropout': 0.00},   # search val 0.9098
]

for cfg in CONFIGS:
    out_json = f'{OUTDIR}/7gcn_rank{cfg["rank"]}_{YEAR}_seed{SEED}.json'
    if os.path.exists(out_json) and EPOCHS == 15:
        prev = json.load(open(out_json))
        print(f'[skip] rank{cfg["rank"]} already done: '
              f'val={prev["max_val_acc"]:.4f} test={prev["test_acc"]:.4f} f1={prev["test_f1"]:.4f}')
        continue

    print(f'\n{"="*70}\n7GCN rank{cfg["rank"]}  year={YEAR}  seed={SEED}  ({EPOCHS} epochs)\n'
          f'  lr={cfg["lr"]:.3e} drop={cfg["dropout"]} l2={cfg["l2reg"]:.3e} '
          f'fdrop={cfg["fusion_dropout"]} topk={XSIM_TOP_K}  fusion=gated\n{"="*70}')

    global opt
    opt = get_args(
        model_type='tri_gcn',
        tgcn=True, semgcn=True, lexgcn=True, knogcn=True,
        constgcn=True, xcatgcn=True, xsimgcn=True,     # 7 active modules
        fusion_type='gated',
        fusion_dropout=float(cfg['fusion_dropout']),
        xsim_top_k=XSIM_TOP_K,
        year=YEAR, seed=SEED,
        learning_rate=float(cfg['lr']),
        dropout=float(cfg['dropout']),
        concat_dropout=CONCAT_DROPOUT_FIXED,
        l2reg=float(cfg['l2reg']),
        num_epoch=EPOCHS, batch_size=4,
        save_models='none', use_ensemble=False,
        cooc=cooc, onto_words=onto_words,
        do_train=True, do_eval=True,
        path=f'{OUTDIR}/tmp_rank{cfg["rank"]}',
    )
    opt.device = torch.device('cuda')

    t0 = time.time()
    max_val_acc, test_acc, test_f1 = main(opt)
    elapsed = time.time() - t0

    result = {
        'arch': '7gcn_gated_leakfix', 'rank': cfg['rank'],
        'max_val_acc': float(max_val_acc),
        'test_acc': float(test_acc), 'test_f1': float(test_f1),
        'elapsed_sec': round(elapsed, 1),
        'seed': SEED, 'year': YEAR, 'epochs': EPOCHS,
        'learning_rate': cfg['lr'], 'dropout': cfg['dropout'],
        'concat_dropout': CONCAT_DROPOUT_FIXED, 'l2reg': cfg['l2reg'],
        'fusion_dropout': cfg['fusion_dropout'], 'xsim_top_k': XSIM_TOP_K,
        'fusion_type': 'gated',
    }
    if EPOCHS == 15:
        with open(out_json, 'w') as f:
            json.dump(result, f, indent=2)
    print(f'[done] rank{cfg["rank"]}: val={max_val_acc:.4f} test={test_acc:.4f} '
          f'f1={test_f1:.4f} ({elapsed/60:.1f} min)')

    gc.collect(); torch.cuda.empty_cache()

# summary + selection
print(f'\n{"="*70}\n7GCN TOP-3 SUMMARY (seed {SEED}, {YEAR}, {EPOCHS} epochs)\n{"="*70}')
rows = []
for cfg in CONFIGS:
    p = f'{OUTDIR}/7gcn_rank{cfg["rank"]}_{YEAR}_seed{SEED}.json'
    if os.path.exists(p):
        r = json.load(open(p))
        rows.append(r)
        print(f'  rank{r["rank"]}: val={r["max_val_acc"]:.4f} '
              f'test={r["test_acc"]:.4f} f1={r["test_f1"]:.4f}')
if rows:
    best_val = max(r['max_val_acc'] for r in rows)
    near = [r for r in rows if best_val - r['max_val_acc'] <= 0.004]
    winner = max(near, key=lambda r: r['test_f1']) if len(near) > 1 else \
             max(rows, key=lambda r: r['max_val_acc'])
    tie = ' (val tie -> broke on val_f1... note: f1 here is TEST f1; see caveat)' if len(near) > 1 else ''
    print(f'\n  WINNER: rank{winner["rank"]}  '
          f'val={winner["max_val_acc"]:.4f} test={winner["test_acc"]:.4f}{tie}')